In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, contains, when, sum, avg, round, max, row_number, desc
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType
import requests
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
import pandas as pd
from pyjstat import pyjstat


In [2]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [3]:
spark = SparkSession.builder.appName("Dairy_Project").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/11 13:38:07 WARN Utils: Your hostname, Benjamins-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.35 instead (on interface en0)
26/04/11 13:38:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/11 13:38:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [15]:
def get_eurostat_simple(year : int):

    year = str(year)

    spark = SparkSession.builder.appName("EurostatData").getOrCreate()

    url = f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/apro_mk_pobta?time={year}"


    dataset = pyjstat.Dataset.read(url)
    df_pandas = dataset.write('dataframe')


    df_spark = spark.createDataFrame(df_pandas)

    df = df_spark.dropna()

    return df

df = get_eurostat_simple(2024)

In [16]:
df.count()

16340

In [17]:
df = df.dropna()

In [18]:
df.count()

4875

In [21]:
df.show(5)

+--------------+---------------------------------------------+---------------+-------------------------------+----+---------+
|Time frequency|Dairy and other animal products (except meat)|   Item of milk|Geopolitical entity (reporting)|Time|    value|
+--------------+---------------------------------------------+---------------+-------------------------------+----+---------+
|        Annual|                               Fresh products|Fat content (t)|                        Belgium|2024| 98577.34|
|        Annual|                               Fresh products|Fat content (t)|                       Bulgaria|2024|   5887.6|
|        Annual|                               Fresh products|Fat content (t)|                        Czechia|2024| 39476.34|
|        Annual|                               Fresh products|Fat content (t)|                        Denmark|2024|  27801.2|
|        Annual|                               Fresh products|Fat content (t)|                        Germany|2024|404

In [24]:
df.select(col("Geopolitical entity (reporting)")).distinct().show()

+-------------------------------+
|Geopolitical entity (reporting)|
+-------------------------------+
|                         Sweden|
|                        Germany|
|                         France|
|                         Greece|
|                       Slovakia|
|                        Belgium|
|           European Union - ...|
|                        Albania|
|                        Finland|
|                        Türkiye|
|                          Malta|
|                        Croatia|
|                          Italy|
|                      Lithuania|
|                         Norway|
|                          Spain|
|                        Czechia|
|                        Denmark|
|                        Ireland|
|                         Cyprus|
+-------------------------------+
only showing top 20 rows


In [25]:
df.select(col("Item of milk")).distinct().show()

+--------------------+
|        Item of milk|
+--------------------+
|     Fat content (t)|
|Products obtained...|
|Utilization of wh...|
|Utilization of sk...|
| Protein content (t)|
+--------------------+

